# Spam Email Classification with PyTorch Lightning (Colab)

This notebook mirrors the local demo. The only intended differences are package install, Kaggle token upload, and a few Colab-friendly runtime defaults.

## Colab note

Place this repo under `/content/intd-491-ML-Demo` by cloning or uploading it before running the notebook.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo_candidates = [Path('/content/intd-491-ML-Demo'), Path.cwd(), Path.cwd().parent]
repo_root = next((path for path in repo_candidates if (path / 'requirements.txt').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not find the repo root with requirements.txt. Clone or upload the repo first.')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(repo_root / 'requirements.txt')])
print(f'Installed requirements from {repo_root / "requirements.txt"}')

In [ ]:
from pathlib import Path
import sys

repo_candidates = [Path('/content/intd-491-ML-Demo'), Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((path for path in repo_candidates if (path / 'src').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate the repo root containing src/.')

for path in (REPO_ROOT / 'src', REPO_ROOT / 'scripts'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f'REPO_ROOT = {REPO_ROOT}')

## Kaggle token setup

You can either pre-place `~/.kaggle/kaggle.json`, or upload it in the next cell and let Colab install it for this runtime.

In [ ]:
from pathlib import Path
from google.colab import files

uploaded = files.upload()
if uploaded:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_path = kaggle_dir / 'kaggle.json'
    name, content = next(iter(uploaded.items()))
    kaggle_path.write_bytes(content)
    kaggle_path.chmod(0o600)
    print(f'Saved {name} to {kaggle_path}')
else:
    print('No file uploaded. Skip this step if ~/.kaggle/kaggle.json already exists.')

In [ ]:
import torch

from spam_lightning.config import ProjectConfig
from spam_lightning.utils.paths import ensure_dir

defaults = ProjectConfig()
KAGGLE_DATASET_SLUG = 'harshsinha1234/email-spam-classification'
RAW_DIR = ensure_dir(REPO_ROOT / defaults.paths.raw_dir)
PROCESSED_DIR = ensure_dir(REPO_ROOT / defaults.paths.processed_dir)
ARTIFACTS_DIR = ensure_dir(REPO_ROOT / defaults.paths.artifacts_dir)
LOGS_DIR = ensure_dir(REPO_ROOT / defaults.paths.logs_dir)
LABEL_MAP_OVERRIDE = {}
SEED = defaults.preprocess.seed
COLAB_NUM_WORKERS = 2
COLAB_PRECISION = '16-mixed' if torch.cuda.is_available() else '32-true'

print(f'Precision = {COLAB_PRECISION}')
print('Set KAGGLE_DATASET_SLUG to a real dataset slug before downloading.')

In [ ]:
from download_kaggle import download_dataset

tabular_files = download_dataset(dataset=KAGGLE_DATASET_SLUG, raw_dir=RAW_DIR)
tabular_files

In [ ]:
import pandas as pd
from spam_lightning.data.preprocessing import discover_tabular_files, select_input_file

all_tabular_files = discover_tabular_files(RAW_DIR)
selected_file = select_input_file(RAW_DIR, None)
if selected_file.suffix.lower() == '.tsv':
    preview_df = pd.read_csv(selected_file, sep='	', nrows=5)
else:
    preview_df = pd.read_csv(selected_file, nrows=5)

print('Discovered files:')
for path in all_tabular_files:
    print(f' - {path}')
print(f'Selected file: {selected_file}')
print(f'Columns: {preview_df.columns.tolist()}')
preview_df.head()

In [ ]:
from spam_lightning.data.preprocessing import preprocess_dataset

preprocess_result = preprocess_dataset(
    raw_dir=RAW_DIR,
    out_dir=PROCESSED_DIR,
    label_map=LABEL_MAP_OVERRIDE or None,
    seed=SEED,
    dataset_slug=KAGGLE_DATASET_SLUG,
)
preprocess_result

In [ ]:
import json
import pandas as pd
from IPython.display import display

stats = json.loads((PROCESSED_DIR / 'stats.json').read_text(encoding='utf-8'))
print(json.dumps(stats, indent=2))

ratio_df = pd.DataFrame(
    {
        'split': list(stats['spam_ratio'].keys()),
        'spam_ratio': list(stats['spam_ratio'].values()),
        'count': [stats['split_sizes'][split] for split in stats['spam_ratio'].keys()],
    }
)
display(ratio_df)

In [ ]:
from spam_lightning.data.datamodule import SpamDataModule

datamodule = SpamDataModule(
    data_dir=PROCESSED_DIR,
    batch_size=defaults.data.batch_size,
    num_workers=COLAB_NUM_WORKERS,
    pin_memory=False,
    lowercase=stats['lowercase'],
    min_freq=defaults.data.min_freq,
    max_vocab_size=defaults.data.max_vocab_size,
)
datamodule.setup('fit')
print(f'Vocab size: {datamodule.vocab_size}')

In [ ]:
tokens, offsets, labels = next(iter(datamodule.train_dataloader()))
print(f'tokens.shape = {tuple(tokens.shape)}')
print(f'offsets.shape = {tuple(offsets.shape)}')
print(f'labels.shape = {tuple(labels.shape)}')
print('First offsets:', offsets[:10].tolist())

In [ ]:
import json
import pytorch_lightning as L
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

from spam_lightning.models.lit_model import SpamLitModule
from spam_lightning.utils.seed import set_global_seed

set_global_seed(SEED)
vocab_path = ARTIFACTS_DIR / 'vocab.json'
datamodule.save_vocab(vocab_path)

model = SpamLitModule(
    vocab_size=datamodule.vocab_size,
    embed_dim=defaults.model.embed_dim,
    learning_rate=defaults.model.learning_rate,
    pad_index=datamodule.pad_index,
)
checkpoint_callback = ModelCheckpoint(
    monitor='val_f1',
    mode='max',
    save_top_k=1,
    filename='best',
    dirpath=str(ARTIFACTS_DIR),
)
trainer = L.Trainer(
    accelerator='auto',
    devices='auto',
    max_epochs=defaults.train.max_epochs,
    deterministic=False,
    precision=COLAB_PRECISION,
    callbacks=[
        checkpoint_callback,
        EarlyStopping(monitor='val_f1', mode='max', patience=3),
        LearningRateMonitor(logging_interval='epoch'),
    ],
    logger=CSVLogger(save_dir=str(LOGS_DIR), name='spam_lightning'),
)
trainer.fit(model, datamodule=datamodule)

BEST_CKPT_PATH = checkpoint_callback.best_model_path
config_payload = {
    'dataset_slug': KAGGLE_DATASET_SLUG,
    'selected_raw_file': stats.get('source_file'),
    'text_col': stats.get('text_col'),
    'label_col': stats.get('label_col'),
    'label_mapping': stats.get('label_mapping', {}),
    'split_ratios': stats.get('split_ratios'),
    'seed': SEED,
    'lowercase': stats.get('lowercase', True),
    'vocab': {
        'min_freq': defaults.data.min_freq,
        'max_vocab_size': defaults.data.max_vocab_size,
        'vocab_path': str(vocab_path.resolve()),
    },
    'model': {
        'embed_dim': defaults.model.embed_dim,
        'learning_rate': defaults.model.learning_rate,
        'vocab_size': datamodule.vocab_size,
        'pad_index': datamodule.pad_index,
    },
    'trainer': {
        'max_epochs': defaults.train.max_epochs,
        'precision': COLAB_PRECISION,
        'deterministic': False,
        'batch_size': defaults.data.batch_size,
        'num_workers': COLAB_NUM_WORKERS,
        'pin_memory': False,
    },
    'artifacts': {
        'best_ckpt': BEST_CKPT_PATH,
        'vocab_json': str(vocab_path.resolve()),
        'config_json': str((ARTIFACTS_DIR / 'config.json').resolve()),
    },
}
(ARTIFACTS_DIR / 'config.json').write_text(json.dumps(config_payload, indent=2), encoding='utf-8')
print(f'Best checkpoint path: {BEST_CKPT_PATH}')

In [ ]:
BEST_CKPT_PATH

In [ ]:
test_results = trainer.test(model, datamodule=datamodule, ckpt_path='best')
test_results

In [ ]:
import torch
from spam_lightning.data.preprocessing import clean_text
from spam_lightning.data.text_utils import load_vocab, regex_tokenize
from spam_lightning.models.lit_model import SpamLitModule

sample_text = 'free money!!! click now to claim your prize'
best_model = SpamLitModule.load_from_checkpoint(BEST_CKPT_PATH)
best_model.eval()
vocab = load_vocab(ARTIFACTS_DIR / 'vocab.json')
normalized_text = clean_text(sample_text, lowercase=bool(stats['lowercase']))
token_ids = [vocab.lookup_index(token) for token in (regex_tokenize(normalized_text, lowercase=False) or ['<unk>'])]

with torch.no_grad():
    probability = torch.sigmoid(
        best_model(
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor([0], dtype=torch.long),
        )
    ).item()

predicted_label = 'spam' if probability >= 0.5 else 'ham'
print(f'Text: {sample_text}')
print(f'Spam probability: {probability:.4f}')
print(f'Predicted label: {predicted_label}')

In [ ]:
print('Artifacts:')
for path in sorted(ARTIFACTS_DIR.glob('*')):
    print(f' - {path.name}')